# Week 5 · Demo — Pairwise & the Biases That Lie to You 🎭
### "Which of these two is better?" — a more reliable question, with traps

Absolute scoring ("rate this 1–4") makes a judge anchor inconsistently. **Pairwise** — "is A or B better?" — is a question judges answer far more reliably, because relative judgments are easier than absolute ones. **If your cohort adopts one eval pattern, make it pairwise.**

But pairwise has traps: the judge carries **biases** — and this notebook makes two of them *visible and reproducible*, then shows which mitigation catches which.

**How to use this notebook:** run top to bottom. It runs **offline** with a stand-in judge that *deliberately simulates* position bias and length bias (so you can see them deterministically), or **live** against `gpt-4o`.

**Where this goes in your app:** the seed of `src/eval/pairwise.py`.

> ⚠️ **About the fake judge:** a real `gpt-4o` judge exhibits these biases *emergently*. Our offline stand-in *simulates* them on purpose so the lesson is reproducible without a key. The biases are real; the simulation just makes them show up on demand.

## 0 · Setup — three *equally-correct* answers

The key to seeing bias is to compare answers that are **equal in quality** but differ in *surface features* (wording, length). If the judge still prefers one, that preference is **bias**, not quality.

- **A** — concise, correct (states 3 days + manager approval)
- **B** — concise, correct, *different wording* (near-identical quality to A)
- **C** — correct, but **much longer** (same facts as A, wrapped in padding)

All three contain the same required facts, so a *fair* judge should call A≈B a tie, and A≈C a tie. Watch what a *biased* judge does instead.

In [ ]:
import os

USE_FAKE = True          # flip to False (+ OPENAI_API_KEY) for the real gpt-4o judge
JUDGE_MODEL = "gpt-4o"
MUST = ["3 days", "manager"]     # the facts a correct answer must contain

answers = {
"A": "You may work remotely up to 3 days per week, with manager approval.",
"B": "Remote work is allowed for up to 3 days weekly, provided your manager approves.",
"C": ("Our remote-work policy is designed to give employees flexibility while keeping teams "
      "aligned. Under the policy, you are permitted to work remotely for up to 3 days per week, "
      "and you should be sure to obtain approval from your manager beforehand so your team has "
      "adequate coverage on the days you are out of office."),
}
for k, v in answers.items():
    print(f"{k}: {len(v.split()):2d} words | {v[:60]}...")

## 1 · The stand-in judge (simulates the biases so we can see them)

Our offline judge computes a hidden "perceived quality" for each answer:
- **true quality** from the required facts (all three answers are equally correct here), plus
- a **length bonus** — this is the *simulated length bias* (the judge reads length as effort).

And when two answers are **close**, it breaks the tie by preferring **whichever it saw first** — the *simulated position bias*.

In [ ]:
TIE = 0.2   # answers within this "perceived quality" gap are a coin-flip -> position decides

def _perceived(text):
    facts = sum(1 for m in MUST if m.lower() in text.lower())
    quality = 3 * facts / len(MUST)                       # 0..3  (all our answers = 3.0)
    length_bonus = min(len(text.split()) / 50, 1.0) * 0.5 # 0..0.5  (SIMULATED length bias)
    return quality + length_bonus

def _judge_pick(first_text, second_text):
    """Returns 'first' or 'second' — which POSITION the judge prefers."""
    qf, qs = _perceived(first_text), _perceived(second_text)
    if abs(qf - qs) < TIE:            # close call -> SIMULATED position bias: prefer first
        return "first"
    return "first" if qf > qs else "second"

def pairwise(first, second):
    """first, second are (label, text). The judge sees `first` then `second`. Returns winning label."""
    if USE_FAKE:
        pick = _judge_pick(first[1], second[1])
    else:
        pick = _real_pairwise(first[1], second[1])
    return first[0] if pick == "first" else second[0]

def _real_pairwise(first_text, second_text):
    import json
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    tool = {"type": "function", "function": {"name": "pick", "parameters": {"type": "object",
            "properties": {"winner": {"type": "string", "enum": ["first", "second"]}},
            "required": ["winner"]}}}
    resp = client.chat.completions.create(model=JUDGE_MODEL, temperature=0,
        messages=[{"role": "system", "content": "Which answer better answers the question? Reply with the tool."},
                  {"role": "user", "content": f"FIRST:\n{first_text}\n\nSECOND:\n{second_text}"}],
        tools=[tool], tool_choice={"type": "function", "function": {"name": "pick"}})
    return json.loads(resp.choices[0].message.tool_calls[0].function.arguments)["winner"]

print("perceived quality:  A=%.2f  B=%.2f  C=%.2f" % (
    _perceived(answers["A"]), _perceived(answers["B"]), _perceived(answers["C"])))

Note the perceived scores: A and B are essentially equal; C is *higher only because it's longer*, even though it says the same thing. A fair judge would tie all three. Let's see what ours does.

## 2 · Position bias, live — the winner flips when you swap the order

A and B are equally good. Ask the judge twice, swapping who goes first.

In [ ]:
A, B, C = ("A", answers["A"]), ("B", answers["B"]), ("C", answers["C"])

print("pairwise(A, B) ->", pairwise(A, B), " (A shown first)")
print("pairwise(B, A) ->", pairwise(B, A), " (B shown first)")

**The winner flipped.** A won when shown first; B won when shown first. Nothing about the *answers* changed — only their **order** did. That's **position bias**: on a close call, the judge favors whatever it saw first. If your eval harness always puts your new prompt in slot A, position bias will hand it a win it didn't earn.

## 3 · Mitigation for position bias — flip and aggregate

The fix is cheap: run **both orders** and only declare a winner if it wins **both times**. If the winner depends on order, it's not a real winner — call it a tie.

In [ ]:
def pairwise_consistent(x, y):
    w1 = pairwise(x, y)     # x first
    w2 = pairwise(y, x)     # y first
    if w1 == w2:
        return w1                       # same winner regardless of order -> trustworthy
    return "TIE (order-dependent)"      # flipped -> position bias, no real winner

print("A vs B, both orders ->", pairwise_consistent(A, B))

Now the harness reports the honest answer: **A and B are a tie** — the earlier "A wins" was an artifact of order. This one habit — **always flip position** — is the single most important pairwise discipline. It costs one extra call and saves you from shipping changes that only "won" because of their slot.

## 4 · Length bias, live — and why the flip does *not* catch it

Now compare **A (concise)** against **C (verbose)**. Remember: they contain the *same facts* — C is just padded. A fair judge ties them.

In [ ]:
print("pairwise(A, C) ->", pairwise(A, C), " (A first)")
print("pairwise(C, A) ->", pairwise(C, A), " (C first)")
print("flip-and-aggregate ->", pairwise_consistent(A, C))

**C wins in *both* orders** — so flip-and-aggregate reports C as a clean, "trustworthy" winner. But C is **not** better; it's just **longer**. This is **length bias**, and notice the uncomfortable lesson:

> **Flip-and-aggregate fixes *position* bias, but it does *nothing* for *length* bias.** A consistent winner can still be a biased winner.

Length bias needs *different* mitigations: a **rubric that explicitly rewards conciseness** (penalize padding), a **strong judge** (better at seeing through verbosity), and **spot-checks**. No single trick removes all bias — you stack defenses.

## 5 · The third bias — self bias (named, not demo'd)

The third bias needs two *different* models to show, so we just name it: **self bias** — a judge tends to prefer answers written in *its own model's style*. If you generate answers with `gpt-4o` and also judge with `gpt-4o`, the judge subtly favors them. Mitigation: judge with a *different* strong model than the one that wrote the answers when you can, and always spot-check.

## 6 · The mitigations, together

You don't eliminate bias — you **bound** it, by stacking cheap defenses:

| bias | what it does | mitigation |
|---|---|---|
| **position** | first answer wins on close calls | **flip position** and aggregate (Section 3) |
| **length** | longer answer wins | rubric that rewards conciseness + strong judge |
| **self** | prefers its own model's style | judge with a different model + spot-check |
| *(all)* | confidently wrong scores | **strong judge** (`gpt-4o`, never mini) + **spot-check 10%** |

**The line to remember:** *a weak judge with no mitigations is worse than no judge — it's confidently wrong. A strong judge with position-flip and spot-checks is genuinely useful.*

## 7 · Your turn — experiment

1. **Go live.** `USE_FAKE = False` + key. Does the real `gpt-4o` show position bias on A vs B? (Often subtler than our simulation — run each order a few times.)
2. **Make C actually better**, not just longer — add a genuinely useful caveat. Now C winning is *correct*. The skill is telling "better" from "longer."
3. **Tune the tie threshold** (`TIE`) — widen it and more comparisons become position-decided. This models a judge that's worse at fine distinctions.
4. **Your domain.** Swap in two of your capstone answers of similar quality and check for order-dependence with `pairwise_consistent`.

## Fit it into your app 🔧

This is the core of **`src/eval/pairwise.py`**:
- `pairwise(first, second)` — one comparison (real path = `gpt-4o` tool-call picking `first`/`second`).
- `pairwise_consistent(x, y)` — **always flip and aggregate**; this is the function `scripts/run_pairwise.py` (Lab Step 4) actually calls.

Lab Step 4: write a **v2** of your answer prompt, then run `pairwise_consistent` over your golden questions comparing v1 vs v2 answers from `/ask`, and document the winner in `docs/prompt-pairwise-001.md` — *with position-flip*, so the winner is real.

**The one-line takeaway:** *pairwise is the most reliable eval question — but only if you flip position; and even then, a consistent winner can still be a biased one, so you stack mitigations and spot-check.*